[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/GEE-Courses/blob/master/docs/gee_intro/ImageCollection/reducing_image_collection.ipynb)

**Reducing an ImageCollection**

To composite images in an `ImageCollection`, use `imageCollection.reduce()`. This will composite all the images in the collection to a single image representing, for example, the min, max, mean or standard deviation of the images. (See the Reducers section for more information about reducers). For example, to create a median value image from a collection:

## Create an interactive map

In [ ]:
import ee
import geemap

In [ ]:
Map = geemap.Map()

## Compute a median image

In [ ]:
# Load a Landsat 8 collection for a single path-row.
collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filter(ee.Filter.eq('WRS_PATH', 44))
    .filter(ee.Filter.eq('WRS_ROW', 34))
    .filterDate('2014-01-01', '2015-01-01')
)

# Compute a median image and display.
median = collection.median()
Map.setCenter(-122.3578, 37.7726, 12)
Map.addLayer(median, {'bands': ['B4', 'B3', 'B2'], 'max': 0.3}, 'median')

In [ ]:
collection.size().getInfo()

In [ ]:
collection.aggregate_array("system:id").getInfo()

## Use median reducer

At each location in the output image, in each band, the pixel value is the median of all unmasked pixels in the input imagery (the images in the collection). In the previous example, `median()` is a convenience method for the following call:

In [ ]:
# Reduce the collection with a median reducer.
median = collection.reduce(ee.Reducer.median())

# Display the median image.
Map.addLayer(
    median,
    {'bands': ['B4_median', 'B3_median', 'B2_median'], 'max': 0.3},
    'also median',
)
Map

## Create an image composite

In [ ]:
states = ee.FeatureCollection('TIGER/2018/States')
Map.addLayer(states, {}, "US States")

In [ ]:
ca = states.filter(ee.Filter.eq("NAME", "California"))
Map.addLayer(ca, {}, "California")

In [ ]:
collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterBounds(ca)
    .filterDate('2020-01-01', '2021-01-01')
)

In [ ]:
collection.size().getInfo()

In [ ]:
image = collection.median().clip(ca)
Map.addLayer(image, {'bands': ['B4', 'B3', 'B2'], 'max': 0.3}, 'Landsat 2020')

## AKB added: max and min for Half Dome (expect to see snow signal)

In [ ]:
#Grok says Half Dome is path 42, row 34.
collectionHD = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filter(ee.Filter.eq('WRS_PATH', 42))
    .filter(ee.Filter.eq('WRS_ROW', 34))
    .filterDate('2014-01-01', '2015-01-01')
)
collectionHD.size().getInfo()

In [ ]:
collectionHD.aggregate_array("system:id").getInfo()

In [ ]:
#alternate method to select images from point instead of path/row
collectionHDalt = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(ee.Geometry.Point(-119.611, 37.737))
)
collectionHDalt.size().getInfo() 
#CONCLUSION: gives twice as many images since it has another row - different dates for path 43 than 42. 
#Maybe could have a point in overlapping rows which would give same dates (essentially redundant data?)

In [ ]:
collectionHDalt.aggregate_array("system:id").getInfo()

In [ ]:
collectionHDcloud = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(ee.Geometry.Point(-119.611, 37.737))
    .filterMetadata('CLOUD_COVER', 'less_than', 10)
    #.sort("CLOUD_COVER")
)
collectionHDcloud.size().getInfo() 

In [ ]:
collectionHDcloud.aggregate_array("system:id").getInfo()

In [ ]:
#plot first image
imageHD = collectionHD.first()
geemap.image_props(imageHD).getInfo()

In [ ]:
Map.addLayer(imageHD, {}, "Half Dome first image")
Map.setCenter(-119.611, 37.737, 12)
Map

In [ ]:
# Reduce the collection with a median reducer.
medianHD = collectionHD.reduce(ee.Reducer.median())
#medianHD.getInfo()

In [ ]:
# Display the median image.
Map.addLayer(
    medianHD,
    {'bands': ['B4_median', 'B3_median', 'B2_median'], 'max': 0.3},
    'medianHD',
)
Map

In [ ]:
# Reduce the collection with a max reducer.
maxHD = collectionHD.reduce(ee.Reducer.max())
maxHD.getInfo()

In [ ]:
# Reduce the collection with a min reducer.
minHD = collectionHD.reduce(ee.Reducer.min())
minHD.getInfo()

In [ ]:
minmaxHD=collectionHD.reduce(ee.Reducer.minMax())
minmaxHD.getInfo()

In [ ]:
minmaxHDcloud=collectionHDcloud.reduce(ee.Reducer.minMax())
minmaxHDcloud.getInfo()

In [ ]:
Map.addLayer(
    maxHD,
    {'bands': ['B4_max', 'B3_max', 'B2_max'], 'max': 0.3},
    'maxHD',
)
Map.addLayer(
    minHD,
    {'bands': ['B4_min', 'B3_min', 'B2_min'], 'max': 0.3},
    'minHD',
)
Map.addLayer(
    minmaxHD,
    {'bands': ['B4_max', 'B3_max', 'B2_max'], 'max': 0.3},
    'minmaxHD max',
)
Map.addLayer(
    minmaxHD,
    {'bands': ['B4_min', 'B3_min', 'B2_min'], 'max': 0.3},
    'minmaxHD min',
)
Map.addLayer(
    minmaxHDcloud,
    {'bands': ['B4_max', 'B3_max', 'B2_max'], 'max': 0.3},
    'minmaxHDcloud max',
)
Map.addLayer(
    minmaxHDcloud,
    {'bands': ['B4_min', 'B3_min', 'B2_min'], 'max': 0.3},
    'minmaxHDcloud min',
)
Map
#CONCLUSION: max and min are the same with two methods of calculating. Data in maxmin more compact.
#CONCLUSION: max is just clouds
#CONCLUSION: the low cloud version of max reveals snow, and some wildfire smoke. 
#CONCLUSION: The all cloud min includes significant cloud shadows that are not present in the low cloud min.